## Gold Facts

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [0]:
from youtube_gold.facts import build_fact_source

dim_channel = spark.table(gold["dim_channel"])

fact_src_df = (
    build_fact_source(scoped_silver(), dim_channel)
    .withColumn("_updated_at", F.current_timestamp())
)

In [0]:
from youtube_gold.facts import build_fact_source

checks = {
    "channel_key not found in dim_channel": find_missing_keys(fact_src_df),
    "category_id not found in dim_category": find_orphans(
        fact_src_df, spark.table(gold["dim_category"]), "category_id"),
    "date_key not found in dim_date": find_orphans(
        fact_src_df, spark.table(gold["dim_date"]), "date_key"),
}

for problem, bad_rows in checks.items():
    if bad_rows.limit(1).count() > 0:
        display(bad_rows.limit(20))
        raise ValueError(f"Fact rows with {problem}")

In [0]:
target = DeltaTable.forName(spark, gold["fact_video_daily_stats"])

changed = """
    NOT (
        t.channel_key      <=> s.channel_key      AND
        t.category_id      <=> s.category_id      AND
        t.views            <=> s.views            AND
        t.likes            <=> s.likes            AND
        t.dislikes         <=> s.dislikes         AND
        t.comment_count    <=> s.comment_count    AND
        t.views_delta      <=> s.views_delta      AND
        t.trending_day_num <=> s.trending_day_num
    )
"""
(
    target.alias("t")
    .merge(
        fact_src_df.alias("s"),
        "t.video_id = s.video_id AND t.date_key = s.date_key"
    )
    .whenMatchedUpdateAll(condition=changed)
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

display(
    target.history(1)
    .select(
        "version",
        "timestamp",
        "operation",
        F.col("operationMetrics.numTargetRowsInserted").alias("inserted"),
        F.col("operationMetrics.numTargetRowsUpdated").alias("updated"),
    )
)